In [6]:
import numpy as np
import matplotlib
from matplotlib import pyplot as plot
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
import sklearn.metrics
from sklearn.metrics import explained_variance_score, mean_squared_error, r2_score # https://scikit-learn.org/stable/auto_examples/linear_model/plot_ols.html
import statsmodels.api as sm
import pandas
import matplotlib.pyplot as plt
import scipy as sp
from scipy.odr import *
import random
import operator
import os

# For 3d plots. This import is necessary to have 3D plotting below
from mpl_toolkits.mplot3d import Axes3D

# For statistics. Requires statsmodels 5.0 or more
from statsmodels.formula.api import ols
from statsmodels.formula.api import wls
# Analysis of Variance (ANOVA) on linear models
from statsmodels.stats.anova import anova_lm

# Adapted from:
# https://www.datasciencecentral.com/profiles/blogs/linear-regression-in-python-use-of-numpy-scipy-and-statsmodels,
# https://realpython.com/linear-regression-in-python/#multiple-linear-regression

In [7]:
M0StarData = np.transpose(np.asarray(np.genfromtxt('ObservablePlus_FundMStarData.csv',delimiter=',',skip_header=0))) # No header, so don't skip
M1StarData = np.transpose(np.asarray(np.genfromtxt('ObservablePlus_FirstOvertoneMStarData.csv',delimiter=',',skip_header=1))) # Has header, so skip
C0StarData = np.transpose(np.asarray(np.genfromtxt('ObservablePlus_FundamentalCStarData.csv',delimiter=',',skip_header=1))) # Has header, so skip
C1StarData = np.transpose(np.asarray(np.genfromtxt('ObservablePlus_FirstOvertoneCStarData.csv',delimiter=',',skip_header=0))) # No header, so don't skip

In [8]:
MdotIndex=5 # Column of imported files that has Mdot data
MdotErrorIndex=6 # Absolute error in Mdot
#MdotIndex=2 # Column of imported files that has Mdot_dust Data
LIndex=8 # Column of imported files that has LBol data
LErrorIndex=9 # Absolute error in L
PIndex=13 # Column of imported files that has Period data
MIndex=22 # Column of imported files that has Mass data
RIndex=23 # Column of imported files that has Radius data
TeffObsvIndex=12 # Column of imported files that has Observed Teff Data
TeffCalcIndex=24 # Column of imported files that has Calculated Teff Data

# For Qx Method
MdotIndex_sorting=4
LIndex_sorting=2
PIndex_sorting=3
MIndex_sorting=0
RIndex_sorting=1
TeffObsvIndex_sorting=5
TeffCalcIndex_sorting=6
MdotperMIndex_sorting=7


New Method

M0 Case:

logR(logL, logP) = 0.289 + 0.417logL + 0.195logP

logM(logL,logP) = -2.112 + 1.085logL - 0.776logP

M1 Case:

logR(logL, logP) = 0.284 + 0.395logL + 0.272logP

logM(logL,logP) = -2.093 + 1.174logL - 1.083logP

C0 Case:

logR(logL, logP) = 0.367 + 0.384logL + 0.234logP

logM(logL,logP) = -1.859 + 0.951logL - 0.631logP

C1 Case:

logR(logL, logP) = 0.365 + 0.351logL + 0.334logP

logM(logL,logP) = -1.854 + 1.041logL - 0.900logP


In [10]:
# Take W = B/C, where logM ~ A+BlogL-ClogP (see above markdown cell for values)
WM0 = 1.085/0.776
WM1 = 1.174/1.083
WC0 = 0.951/0.631
WC1 = 1.041/0.900
print('W_M0=', WM0,'; W_M1=', WM1,'; W_C0=', WC0,'; W_C1=', WC1)

W_M0= 1.3981958762886597 ; W_M1= 1.0840258541089567 ; W_C0= 1.5071315372424723 ; W_C1= 1.1566666666666665


In [54]:
M0StarData[PIndex]

array([144.55 , 240.269, 152.999, ..., 122.564, 135.355, 192.864])

In [58]:
M0StarData[LIndex]**WM0

array([ 75281.91604938, 173973.91503934,  92825.25234163, ...,
        75281.91604938,  83587.6959481 , 214525.9958445 ])

In [56]:
def calculate_log_Q (PData, LData, W):
    Q = PData/(LData**W)
    logQ = np.log10(Q)
    return logQ

In [57]:
calculate_log_Q (M0StarData[PIndex],M0StarData[LIndex],WM0)

array([-2.71667257, -2.8597864 , -2.78297755, ..., -2.78832774,
       -2.79066805, -3.04622876])

In [59]:
np.log10(M0StarData[PIndex])-WM0*np.log10(M0StarData[LIndex])

array([-2.71667257, -2.8597864 , -2.78297755, ..., -2.78832774,
       -2.79066805, -3.04622876])

In [48]:
def calc_rms (data_set):
    sum_of_squares = 0
    for i in range(0,len(data_set)):
        sum_of_squares = sum_of_squares + data_set[i]**2
    rms = np.sqrt(sum_of_squares/len(data_set))
    return rms

def calc_rms_error (predicted_data, obsv_data):
    sum_of_squares_of_error = 0
    if len(predicted_data) != len(obsv_data):
        print ('Warning! Data Sets of Unequal Size! Calculation is invalid!')
    for i in range(0,len(predicted_data)):
        sum_of_squares_of_error = sum_of_squares_of_error + (predicted_data[i] - obsv_data[i])**2
    ms_error = sum_of_squares_of_error/len(predicted_data)
    rms_error = np.sqrt(ms_error)
    return rms_error
    
lum_limits=(2.8,4.5)
per_limits=(1.2,3)
mass_limits=(np.log(0.5),np.log(5))
Q_limits=(-4,4)
mdot_limits=(-9,-4)
radius_limits=(0,3)
resid_limits=(-3,3)

In [49]:
# This function takes a set of stars (already sorted by type and pulsation mode) and fits a mass loss fit using the stars in a given percentile range as the basis for the fit
def mdot_fit_Q_anly_W (data_set, specify_star_type ='', W=1, save_figures = 0):
    if specify_star_type == 'M0':
        save_directory = 'MDot_Fits/Analytic_Q/M0/'
    elif specify_star_type == 'M1':
        save_directory = 'MDot_Fits/Analytic_Q/M1/'
    elif specify_star_type == 'C0':
        save_directory = 'MDot_Fits/Analytic_Q/C0/'
    elif specify_star_type == 'C1':
        save_directory = 'MDot_Fits/Analytic_Q/C1/'
    else:
        save_directory = 'MDot_Fits/Analytic_Q/Other/'
    # Create directories if they do not exist
    
    # Use an algorithm to determine V and W. Defined earlier in the code.
    V = 0
    WRatio = W
    # Restore Unrestricted data so we can plot all of it, not just the data used to make our formula
    DataToFit=data_set # Assign one of the data sets to a new variable. This will save some effort down the line and limit memory usage.
    ## Now take the columns we want from the data set and assign them to their own variables.
    logMData = np.log10(DataToFit[MIndex]) 
    logRData = np.log10(DataToFit[RIndex])
    logLData = np.log10(DataToFit[LIndex])
    logLDataError = np.log10(DataToFit[LErrorIndex])
    logPData = np.log10(DataToFit[PIndex])
    logMdotData = np.log10(DataToFit[MdotIndex])
    logMdotDataError = np.log10(DataToFit[MdotErrorIndex])
    logTeffObsvData = np.log10(DataToFit[TeffObsvIndex])
    logTeffCalcData = np.log10(DataToFit[TeffCalcIndex])
    logPdivLWData = calculate_log_Q (DataToFit[PIndex],DataToFit[LIndex],WRatio)
    
    # Do our first fit for determining a function for Mdot
    ## logMdot ~ logA + c log((P/L)^W) + d logM
    x0 = logPdivLWData
    x1 = logMData
    y = logMdotData
    data = pandas.DataFrame({'logPdivLW': x0, 'logM': x1, 'logMdot': y})
    # Fit the model
    model = wls("logMdot ~ logPdivLW + logM", data).fit()
    
    # Refine the previous fit using an ODR Method
    ## Define a function (four-variable linear in our case) to fit the data with.
    def fitting_func(B, x):
        return B[0]*x[0] + B[1]*x[1] + B[2] # A simple linear fit
    ## Create a model for fitting.
    linear_model = Model(fitting_func)
    # Create a RealData object using our initiated data from above.
    data = RealData((logPdivLWData, logMData), y=logMdotData) # Data: x=logL, y=logR
    ## Set up ODR with the model and data.
    odr = ODR(data, linear_model, beta0=[model._results.params[1],model._results.params[2],model._results.params[0]]) # Use the linear regression as an initial guess
    ## Run the regression.
    outPdivLW_M = odr.run()
    
    # Using the above fit, calculate the resutls for logMdot
    logMdot_PdivLM = outPdivLW_M.beta[0]*logPdivLWData + outPdivLW_M.beta[1]*logMData + outPdivLW_M.beta[2]
    print( 'logMdot =', outPdivLW_M.beta[0], 'log(P/L^W) + ', outPdivLW_M.beta[1], 'logM + ', outPdivLW_M.beta[2])
    
    # Do a fit of calculated Mdot vs observed Mdot for plotting
    x0 = logMdotData
    y = logMdot_PdivLM
    data = pandas.DataFrame({'logMdotData': x0, 'logMdot_PdivLM': y})
    # Fit the model
    model = ols("logMdot_PdivLM ~ logMdotData", data).fit()
    PdivLMLine = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    
    # Plot our initial fit, and see how it compares to observation
    first_fit_mdot_fig = plt.figure()
    SlopeTestNoErrors=PdivLMLine[0] + PdivLMLine[1] *(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]))
    print('Comparison to Observed:','logMdot_calc = (',PdivLMLine[0],') + (',PdivLMLine[1],')*logMdot_obsv')
    plot.scatter(logMdotData, logMdot_PdivLM, s=2, color='green')
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),SlopeTestNoErrors, color='green')
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),color='black')
    plot.xlabel("logMdot_obsv")
    plot.ylabel("logMdot_calc")
    plot.xlim(-10, -4)
    plot.ylim(-10, -4)
    plot.title(specify_star_type+' - No Errors Included')
    if save_figures != 0:
        first_fit_mdot_fig.savefig(save_directory+'calc_vs_obsv_first_fit.png')
    plot.close()
    
    # With that first fit done, we will start improving it by subtracting the residuals from the fit
    ResidualYAxis = logMdot_PdivLM - logMdotData # Calc - Obsv; Gives the distance between the regression line and the data
    PlotTitle = specify_star_type + ' - No Errors Included' # Label the plot with whatever data set we are using
    
    # Fit logP to the residuals in a linear fit
    # First we need to flatten the data: it's 2D layout is not relevent.
    x0 = logPData
    y = ResidualYAxis
    data = pandas.DataFrame({'logPData': x0, 'ResidualYAxis': y})
    model = ols("ResidualYAxis ~ logPData", data).fit()
    ResidualsFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Residual Fit: logMdotCalc-logMdotObsv =', ResidualsFit[0], '+', ResidualsFit[1], 'logP')
    
    # Now, we are going to start comparing the residual to various useful parameters
    ylabel = 'Mdot Residuals' # What is on the y label until we change this value
    
    # NewlogMdotCalc - logMdotobs vs logP
    xdataSet = logPData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logP')
    first_resid_vs_P = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logP")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(per_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        first_resid_vs_P.savefig(save_directory+'Residual_Images/first_resid_vs_logP.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logL
    xdataSet = logLData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logL')
    first_resid_vs_L = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logL")
    plot.ylabel(ylabel)
    plot.xlim(lum_limits)
    plot.ylim(resid_limits)
    plot.title(PlotTitle)
    if save_figures != 0:
        first_resid_vs_L.savefig(save_directory+'Residual_Images/first_resid_vs_logL.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logQ (Q = (P/L^W))
    logQData = np.log10(10**(logPData)/(10**(logLData))**(WRatio))
    xdataSet = logQData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logQ')
    first_resid_vs_Q = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logQ")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(Q_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        first_resid_vs_Q.savefig(save_directory+'Residual_Images/first_resid_vs_logQ.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logM
    xdataSet = logMData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logM')
    first_resid_vs_M = plt.figure()
    plot.scatter(logMData, ResidualYAxis, s=2, color='green')
    plot.xlabel("logM")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(mass_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        first_resid_vs_M.savefig(save_directory+'Residual_Images/first_resid_vs_logM.png')
    plot.close()
    
    # Now, remove the residuals
    NewMdotCalc = outPdivLW_M.beta[0]*logPdivLWData + outPdivLW_M.beta[1]*logMData + outPdivLW_M.beta[2] - ResidualsFit[0] - ResidualsFit[1]*logPData
    ylabel = "NewlogMdotCalc"
    print('NewMdotCalc =', -ResidualsFit[0] + outPdivLW_M.beta[2], '+', outPdivLW_M.beta[0], 'log(P/L^W) + ', outPdivLW_M.beta[1], 'logM - ', ResidualsFit[1], 'logP', '; W =', WRatio )
    # Simplified statement: constants + A log P + B log L + log M
    print('NewMdotCalc =', -ResidualsFit[0] + outPdivLW_M.beta[2], '+ (', outPdivLW_M.beta[0] - ResidualsFit[1],') logP + (', -1*outPdivLW_M.beta[0]*WRatio,') logL + ', outPdivLW_M.beta[1], 'logM'  )
    
    # logMdot ~ logM + logL + logP
    xdataSet = logMdotData
    x0 = xdataSet
    y = NewMdotCalc
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('NewMdotCalc Fit: NewlogMdotCalc =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logMdot_obsv')
    newmdotcalc_vs_mdotobsv = plt.figure()
    plot.scatter(xdataSet, NewMdotCalc, s=2, color='green')
    plot.plot(np.asarray([-9,-7,-6,-4]),np.asarray([-9,-7,-6,-4]),color='black')
    SlopeTestNoErrors=NewMdotCalcFit[0] +NewMdotCalcFit[1] *(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]))
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),SlopeTestNoErrors, color='green')
    plot.xlim(mdot_limits)
    plot.ylim(mdot_limits)
    plot.xlabel("logMdot_obsv")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        newmdotcalc_vs_mdotobsv.savefig(save_directory+'newmdotcalc_vs_mdotobsv.png')
    plot.close()
    
    # Now, we're going to compare residuals
    
    NewResidual = NewMdotCalc - logMdotData # Calc - Obsv; Gives the distance between the regression line and the data
    ylabel = 'Old Residual' # What is on the y label until we change this value
    xdataSet = NewResidual
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Residuals Fit: OldResidual =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'NewResidual')
    second_resid_vs_first_resid = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    SlopeTestNoErrors=NewMdotCalcFit[0] +NewMdotCalcFit[1] *(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]))
    plot.plot(np.asarray([-10,-9,-8,-7,-6,-5,-4,-3]),SlopeTestNoErrors, color='green')
    plot.plot(np.asarray([-4,-1,0,1,4]),np.asarray([-4,-1,0,1,4]),color='black')
    plot.xlim(resid_limits)
    plot.ylim(resid_limits)
    plot.xlabel("New Residual")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        second_resid_vs_first_resid.savefig(save_directory+'Residual_Images/second_resid_vs_first_resid.png')
    plot.close()
    
    ### RMS of Data
    RMSE = calc_rms_error (NewMdotCalc, logMdotData)
    print ('RMS Error:', RMSE)
    print ('STD Dev:', np.std(NewMdotCalc))
    
    ylabel = 'Second Residual'
    ResidualYAxis = NewResidual
    
    # NewlogMdotCalc - logMdotobs vs logP
    xdataSet = logPData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logP')
    second_resid_vs_logP = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logP")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(per_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        second_resid_vs_logP.savefig(save_directory+'Residual_Images/second_resid_vs_logP.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logL
    xdataSet = logLData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logL')
    second_resid_vs_logL = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlabel("logL")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    plot.xlim(lum_limits)
    plot.ylim(resid_limits)
    if save_figures != 0:
        second_resid_vs_logL.savefig(save_directory+'Residual_Images/second_resid_vs_logL.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logQ (Q = (P/L^W))
    logQData = np.log10(10**(logPData)/(10**(logLData))**(WRatio))
    xdataSet = logQData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logQ')
    second_resid_vs_logQ = plt.figure()
    plot.scatter(xdataSet, ResidualYAxis, s=2, color='green')
    plot.xlim(Q_limits)
    plot.ylim(resid_limits)
    plot.xlabel("logQ")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        second_resid_vs_logQ.savefig(save_directory+'Residual_Images/second_resid_vs_logQ.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logM
    xdataSet = logMData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logM')
    second_resid_vs_logM = plt.figure()
    plot.scatter(logMData, ResidualYAxis, s=2, color='green')
    plot.xlim(mass_limits)
    plot.ylim(resid_limits)
    plot.xlabel("logM")
    plot.ylabel(ylabel)
    plot.title(PlotTitle)
    if save_figures != 0:
        second_resid_vs_logM.savefig(save_directory+'Residual_Images/second_resid_vs_logM.png')
    plot.close()
    
    # NewlogMdotCalc - logMdotobs vs logR
    xdataSet = logRData
    x0 = xdataSet
    y = ResidualYAxis
    data = pandas.DataFrame({'fittingData': x0, 'NewMdotCalc': y})
    model = ols("NewMdotCalc ~ fittingData", data).fit()
    NewMdotCalcFit = model._results.params # Defines the quality-of-fit line with array (intercept, slope)
    print('Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv =', NewMdotCalcFit[0], '+', NewMdotCalcFit[1], 'logR')
    second_resid_vs_logR = plt.figure()
    plot.scatter(logRData, NewMdotCalc, s=2, color='green')
    plot.xlim(radius_limits)
    plot.ylim(mdot_limits)
    plot.xlabel("logR")
    plot.ylabel('Mdot_calc_New')
    plot.title(PlotTitle)
    if save_figures != 0:
        second_resid_vs_logR.savefig(save_directory+'Residual_Images/second_resid_vs_logR.png')
    plot.close()
    
    #return 1 # Placeholder. Should change to return values of interest in an array.
    return [-ResidualsFit[0] + outPdivLW_M.beta[2], outPdivLW_M.beta[0] - ResidualsFit[1], -1*outPdivLW_M.beta[0]*WRatio, outPdivLW_M.beta[1] ], NewMdotCalc, logMdotData, [V,WRatio]
    # What data do we need to carry over for a piecewise?: RawMdotData, NewMdotcalc coefficients, NewMdotCalc of input data

In [50]:
mdot_fit_Q_anly_W (M0StarData, specify_star_type = 'M0', W=WM0, save_figures = 1)

logMdot = 199.394352064449 log(P/L^W) +  256.5008292173262 logM +  535.9113509138251
Comparison to Observed: logMdot_calc = ( -6.645258336513979 ) + ( 0.019246721320862636 )*logMdot_obsv
Residual Fit: logMdotCalc-logMdotObsv = 2.6173957163583665 + -1.1925783982312614 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 2.6173957163583665 + -1.1925783982312614 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 1.9504176960637611 + -0.5528993519545824 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.041734548767732686 + -0.015295670148736 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.00010509599179103792 + 0.019710915140123984 logM
NewMdotCalc = 533.2939551974667 + 199.394352064449 log(P/L^W) +  256.5008292173262 logM -  -1.1925783982312614 logP ; W = 1.3981958762886597
NewMdotCalc = 533.2939551974667 + ( 200.58693046268027 ) logP + ( -278.7923608117618 ) logL +  256.5008292173262 logM
NewMdotCalc Fit: NewlogMdotCalc = -5.624034853875545 + 0

([533.2939551974667,
  200.58693046268027,
  -278.7923608117618,
  256.5008292173262],
 array([-6.80984893, -6.59671928, -6.80361485, ..., -6.92036176,
        -6.86976649, -6.7757473 ]),
 array([-6.54925578, -6.93120769, -6.48855794, ..., -6.54925578,
        -6.59501328, -7.22703649]),
 [0, 1.3981958762886597])

In [51]:
mdot_fit_Q_anly_W (M1StarData, specify_star_type = 'M1', W=WM1, save_figures = 1)

logMdot = -4269.910775177076 log(P/L^W) +  -3943.1730531422963 logM +  -8267.899348744202
Comparison to Observed: logMdot_calc = ( -4.342509573082824 ) + ( 0.3708444023672328 )*logMdot_obsv
Residual Fit: logMdotCalc-logMdotObsv = -0.2302930738893188 + 0.1185444254462226 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.2302930738893188 + 0.1185444254462226 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.17807071963838594 + 0.04971743471346429 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 0.0012978293360638923 + 0.0006476328476703666 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 4.5442914717948226e-05 + -0.0006921234778598205 logM
NewMdotCalc = -8267.669055670312 + -4269.910775177076 log(P/L^W) +  -3943.1730531422963 logM -  0.1185444254462226 logP ; W = 1.0840258541089567
NewMdotCalc = -8267.669055670312 + ( -4270.029319602522 ) logP + ( 4628.693675030368 ) logL +  -3943.1730531422963 logM
NewMdotCalc Fit: NewlogMdotCalc = -4.5015

([-8267.669055670312,
  -4270.029319602522,
  4628.693675030368,
  -3943.1730531422963],
 array([-6.43831439, -6.8615637 , -6.50247834, ..., -6.86452648,
        -6.69888189, -7.13528522]),
 array([-6.37316453, -7.28208406, -6.98696292, ..., -7.45234577,
        -6.6461658 , -7.40312775]),
 [0, 1.0840258541089567])

In [52]:
mdot_fit_Q_anly_W (C0StarData, specify_star_type = 'C0', W=WC0, save_figures = 1)

logMdot = 146.7384405372164 log(P/L^W) +  232.0460218503145 logM +  425.9019185986809
Comparison to Observed: logMdot_calc = ( -6.354191330835672 ) + ( 0.021095861044839848 )*logMdot_obsv
Residual Fit: logMdotCalc-logMdotObsv = 6.4292983450240655 + -2.5497553434741045 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 6.4292983450240655 + -2.5497553434741045 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 2.489115794152456 + -0.6540955300872355 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.03480501019326927 + -0.01111099140822705 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.002070726472359464 + 0.01760854422856839 logM
NewMdotCalc = 419.4726202536568 + 146.7384405372164 log(P/L^W) +  232.0460218503145 logM -  -2.5497553434741045 logP ; W = 1.5071315372424723
NewMdotCalc = 419.4726202536568 + ( 149.28819588069052 ) logP + ( -221.15413145941807 ) logL +  232.0460218503145 logM
NewMdotCalc Fit: NewlogMdotCalc = -4.176111301541452 + 0

([419.4726202536568,
  149.28819588069052,
  -221.15413145941807,
  232.0460218503145],
 array([-7.14409023, -6.67525153, -6.24628679, ..., -6.26690693,
        -6.55744505, -6.8057008 ]),
 array([-6.74122305, -6.74122305, -5.43505341, ..., -5.20310433,
        -6.08571873, -7.04225304]),
 [0, 1.5071315372424723])

In [53]:
mdot_fit_Q_anly_W (C1StarData, specify_star_type = 'C1', W=WC1, save_figures = 1)

logMdot = 112.20821052249214 log(P/L^W) +  124.75589782362508 logM +  224.45928751838187
Comparison to Observed: logMdot_calc = ( -6.6733448459190035 ) + ( 0.00044936912246842237 )*logMdot_obsv
Residual Fit: logMdotCalc-logMdotObsv = 4.14351895796569 + -1.858906675798965 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 4.14351895796569 + -1.858906675798965 logP
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = 2.7125798572665056 + -0.7016274157072143 logL
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.08090905418699885 + -0.03548696199019699 logQ
Resid. vs Parameter Fit: logMdot_calc - logMdot_obsv = -0.007805912487193322 + 0.039429957766885675 logM
NewMdotCalc = 220.3157685604162 + 112.20821052249214 log(P/L^W) +  124.75589782362508 logM -  -1.858906675798965 logP ; W = 1.1566666666666665
NewMdotCalc = 220.3157685604162 + ( 114.0671171982911 ) logP + ( -129.78749683768257 ) logL +  124.75589782362508 logM
NewMdotCalc Fit: NewlogMdotCalc = -4.181545016097463 

([220.3157685604162,
  114.0671171982911,
  -129.78749683768257,
  124.75589782362508],
 array([-6.79930533, -6.97266452, -6.54567264, -6.15935341, -6.68231817,
        -6.11666616, -6.88611377, -6.24911018, -7.03309237, -6.55630914,
        -6.77699205, -6.2676636 , -6.84808129, -6.65058413, -6.80490499,
        -6.86801037, -6.76658972, -6.6861828 , -6.96847112, -6.88171706,
        -6.99049943, -6.50387781, -6.67205491, -6.72677376, -6.77283214,
        -6.18447243, -6.58929685, -6.90401924, -6.84299908, -6.55093242,
        -6.63730635, -6.48072656, -6.77576134, -6.530157  , -6.55189158,
        -6.45594082, -6.82375997, -6.89554144, -6.77186899, -6.47105956,
        -6.76199358, -6.76652407, -6.60954415, -6.48663549, -6.68004111,
        -6.58183377, -6.35234022, -6.55405574, -6.52181582, -6.50934535,
        -6.9187466 , -6.6883402 , -6.59815828, -6.61183559, -6.51165498,
        -6.76987927, -6.36813672, -6.74852857, -6.80752669, -6.74018181,
        -6.62122114, -6.76949373, -6